# Political dataset descriptives

Each CSV record contains paired A/B options and yields two identity–option observations. Agreement values are probabilities (%), not response frequencies.

In [1]:
import pandas as pd

pd.options.display.float_format = "{:.2f}".format
pd.options.display.max_rows = 40

data = pd.read_csv("csv_data/pol_data_percent.csv")

In [2]:
coverage = data.groupby("model").agg(
    records=("question", "size"),
    question_texts=("question", "nunique"),
    topics=("topic", "nunique"),
)
coverage["question_option_pairs"] = (
    data.drop_duplicates(["model", "question", "option_A", "option_B"])
    .groupby("model").size()
)
coverage["identity_option_observations"] = 2 * coverage["records"]
coverage

,records,question_texts,topics,question_option_pairs,identity_option_observations
model,,,,,
gpt_3_5,12924,10246,12,12923,25848
gpt_4,13122,10409,12,13121,26244


In [3]:
data.groupby(["topic", "model"]).size().unstack("model").rename_axis(columns=None)

,gpt_3_5,gpt_4
topic,,
Climate Change,1229,1427
Defense and Military,1257,1257
Economic Policies,1407,1407
Globalization,1330,1330
Healthcare,1261,1261
Immigration,1215,1215
Law and Order,1188,1188
Media Freedom,1212,1212
Social Equality and Rights,1221,1221


## Agreement by biography

The opposing-biography condition uses the complement of agreement with the other option, matching the paper's plotting code.

In [4]:
options = pd.concat([
    data[["model", "topic", "identityA", "agree_A", "empty_A", "disagree_B"]]
        .rename(columns={"identityA": "identity", "agree_A": "Agreeing Bio",
                         "empty_A": "No Bio", "disagree_B": "Opposing Bio"}),
    data[["model", "topic", "identityB", "agree_B", "empty_B", "disagree_A"]]
        .rename(columns={"identityB": "identity", "agree_B": "Agreeing Bio",
                         "empty_B": "No Bio", "disagree_A": "Opposing Bio"}),
], ignore_index=True)
conditions = ["Opposing Bio", "No Bio", "Agreeing Bio"]
agreement = options.melt(
    id_vars=["model", "topic", "identity"], value_vars=conditions,
    var_name="biography", value_name="agreement_percent",
)
agreement["biography"] = pd.Categorical(
    agreement["biography"], categories=conditions, ordered=True,
)

In [5]:
agreement.groupby(["model", "biography"], observed=True)["agreement_percent"].agg(
    n="count", mean="mean", sd="std", minimum="min", median="median", maximum="max",
)

n  mean    sd  minimum  median  maximum
model   biography                                                
gpt_3_5 Opposing Bio  25848 21.75 38.94     0.00    0.05   100.00
        No Bio        25848 49.06 47.81     0.00   40.72   100.00
        Agreeing Bio  25848 78.25 38.94     0.00   99.95   100.00
gpt_4   Opposing Bio  26244 25.51 41.82     0.00    0.01   100.00
        No Bio        26244 49.47 48.09     0.00   43.74   100.00
        Agreeing Bio  26244 74.49 41.82     0.00   99.99   100.00

In [6]:
by_identity = options.groupby(["model", "identity"])
identity_summary = by_identity[conditions].mean()
identity_summary.insert(0, "observations", by_identity.size())
identity_summary

observations  Opposing Bio  No Bio  Agreeing Bio
model   identity                                                           
gpt_3_5 authoritarians             1876         10.62   40.87         80.61
        capitalists                1926          3.14   15.69         57.16
        collectivists              2024         31.45   83.34         98.30
        conservatives              1987          2.92    9.43         72.44
        environmentalists          1005         55.16   87.52         95.75
        individualists             2024          1.70   15.02         68.55
        industrialists             1005          4.25   11.03         44.84
        liberals                   1987         27.56   87.19         97.08
        libertarians               1876         19.39   56.47         89.38
        progressives               2149         39.91   86.90         97.20
        secularists                1957         63.71   86.05         93.11
        socialists                 1926         42.84   82.77         96.86
        theocrats                  1957          6.89   12.91         36.29
        traditionalists            2149          2.80   11.81         60.09
gpt_4   authoritarians             2074          9.96   41.01         71.94
        capitalists                1926          3.76   16.61         58.04
        collectivists              2024         44.64   83.60         96.09
        conservatives              1987          3.82    9.73         68.36
        environmentalists          1005         60.93   86.14         92.78
        individualists             2024          3.91   15.13         55.36
        industrialists             1005          7.22   13.84         39.07
        liberals                   1987         31.64   90.13         96.18
        libertarians               2074         28.06   58.31         90.04
        progressives               2149         48.87   86.99         96.65
        secularists                1957         70.80   86.00         92.48
        socialists                 1926         41.96   82.74         96.24
        theocrats                  1957          7.52   13.64         29.20
        traditionalists            2149          3.35    9.43         51.13